In [1]:
from google.colab import drive
drive.mount('/content/drive')

import shutil,os

src="astronomy_exp005_backup"
dst="astronomy_exp005"

if os.path.exists(dst):
    shutil.rmtree(dst)

shutil.copytree(src,dst)

print("RESTORE COMPLETE")
print("Workspace:",dst)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
RESTORE COMPLETE
Workspace: astronomy_exp005


In [2]:
import os

ROOT="astronomy_exp005"
IMG=os.path.join(ROOT,"results","exp034_images")

print("Workspace:", os.path.exists(ROOT))
print("EXP-034 images:", os.path.exists(IMG))

if os.path.exists(IMG):
    files=sorted(os.listdir(IMG))
    print("FITS files:",len([f for f in files if f.lower().endswith((".fits",".fits.gz"))]))
    print("\nActual image files:")
    for f in files:
        if f.lower().endswith((".fits",".fits.gz")):
            print(f)

print("\nKey result files:")
for f in [
    "exp034B_downloads.csv",
    "exp034C_image_photometry.csv",
    "exp034D_artifact_audit.csv",
    "exp034E_image_comparison.csv",
]:
    p=os.path.join(ROOT,"results",f)
    print(f, "OK" if os.path.exists(p) else "MISSING")

Workspace: True
EXP-034 images: True
FITS files: 16

Actual image files:
g1974_rank1_w1.fits
g1974_rank1_w2.fits
g1974_rank2_w1.fits
g1974_rank2_w2.fits
g1974_rank3_w1.fits
g1974_rank3_w2.fits
g1974_rank4_w1.fits
g1974_rank4_w2.fits
g1978_rank1_w1.fits
g1978_rank1_w2.fits
g1978_rank2_w1.fits
g1978_rank2_w2.fits
g1978_rank3_w1.fits
g1978_rank3_w2.fits
g1978_rank4_w1.fits
g1978_rank4_w2.fits

Key result files:
exp034B_downloads.csv OK
exp034C_image_photometry.csv OK
exp034D_artifact_audit.csv MISSING
exp034E_image_comparison.csv MISSING


In [3]:
import os,numpy as np,pandas as pd
from astropy.io import fits

ROOT="astronomy_exp005"
IMG=os.path.join(ROOT,"results","exp034_images")
OUT=os.path.join(ROOT,"results")

C=pd.read_csv(os.path.join(OUT,"exp034C_image_photometry.csv"))
rows=[]

for _,r in C.iterrows():
    path=os.path.join(IMG,r["file"].replace(".gz",""))
    if not os.path.exists(path):
        path=os.path.join(IMG,f"g{int(r.survivor)}_rank{int(r['rank'])}_w{int(r.band)}.fits")
    if not os.path.exists(path):
        raise FileNotFoundError(path)

    with fits.open(path,memmap=False) as h:
        data=np.squeeze(h[0].data).astype(float)

    x,y=float(r.x),float(r.y)
    ny,nx=data.shape
    yy,xx=np.indices(data.shape)
    rr=np.hypot(xx-x,yy-y)

    ap=rr<=3
    ann=(rr>=6)&(rr<=12)
    bgv=data[ann]
    bg=np.nanmedian(bgv)
    mad=1.4826*np.nanmedian(np.abs(bgv-bg))
    flux=np.nansum(data[ap]-bg)
    peak=np.nanmax(data[ap])
    w=data[ap]-bg
    ys,xs=np.where(ap)
    w=np.clip(w,0,None)
    sw=w.sum()
    cx=(xs*w).sum()/sw if sw>0 else np.nan
    cy=(ys*w).sum()/sw if sw>0 else np.nan
    snr=flux/(np.sqrt(np.nansum(np.abs(data[ap]-bg)))+1e-9)

    rows.append([
        int(r.survivor),int(r["rank"]),int(r.band),os.path.basename(path),
        x,y,nx,ny,True,flux,peak,snr,cx,cy,bg,mad
    ])

D=pd.DataFrame(rows,columns=[
    "survivor","rank","band","file","x","y","nx","ny","inside",
    "background_subtracted_flux","peak_pixel","aperture_snr_proxy",
    "centroid_x","centroid_y","background","background_robust_sigma"
])

D.to_csv(os.path.join(OUT,"exp034F_corrected_influential_remeasurements.csv"),index=False)

print("EXP-034F CORRECTED")
print(D.to_string(index=False))
print("\nSaved:",os.path.join(OUT,"exp034F_corrected_influential_remeasurements.csv"))

EXP-034F CORRECTED
 survivor  rank  band                file         x         y  nx  ny  inside  background_subtracted_flux  peak_pixel  aperture_snr_proxy  centroid_x  centroid_y  background  background_robust_sigma
     1974     1     1 g1974_rank1_w1.fits 50.402526 49.779455 101  89    True                  589.809612  131.021973           24.270958         NaN         NaN   24.990196                 4.366206
     1974     1     2 g1974_rank1_w2.fits 50.057645 49.682119 101  89    True                  190.639496  108.069237           12.009670   50.329542   49.831028   74.168976                 5.229052
     1974     2     1 g1974_rank2_w1.fits 49.605717 49.769642 101 101    True                  602.586261   83.161407           24.478670   49.743630   49.809866   23.211053                 3.798492
     1974     2     2 g1974_rank2_w2.fits 50.008220 50.326884 101 101    True                  222.420036   99.602722           13.954918         NaN         NaN   73.787804            

In [4]:
import os,numpy as np,pandas as pd
from astropy.io import fits

ROOT="astronomy_exp005"
IMG=os.path.join(ROOT,"results","exp034_images")
OUT=os.path.join(ROOT,"results")

C=pd.read_csv(os.path.join(OUT,"exp034C_image_photometry.csv"))
rows=[]

for _,r in C.iterrows():
    gid,rank,band=int(r.survivor),int(r["rank"]),int(r.band)
    path=os.path.join(IMG,f"g{gid}_rank{rank}_w{band}.fits")

    if not os.path.exists(path):
        raise FileNotFoundError(path)

    with fits.open(path,memmap=False) as h:
        data=np.squeeze(h[0].data).astype(float)

    x,y=float(r.x),float(r.y)
    ny,nx=data.shape
    yy,xx=np.indices(data.shape)
    rr=np.hypot(xx-x,yy-y)

    ap=(rr<=3)&np.isfinite(data)
    ann=(rr>=6)&(rr<=12)&np.isfinite(data)

    bgv=data[ann]
    bg=np.nanmedian(bgv)
    mad=1.4826*np.nanmedian(np.abs(bgv-bg))

    vals=data[ap]-bg
    flux=np.nansum(vals)
    peak=np.nanmax(data[ap])
    snr=flux/(np.sqrt(np.nansum(np.abs(vals)))+1e-9)

    w=np.clip(vals,0,None)
    ys,xs=np.where(ap)
    sw=w.sum()
    cx=(xs*w).sum()/sw if sw>0 else np.nan
    cy=(ys*w).sum()/sw if sw>0 else np.nan

    rows.append([
        gid,rank,band,os.path.basename(path),x,y,nx,ny,
        flux,peak,snr,cx,cy,bg,mad,
        int(ap.sum()),int(ann.sum())
    ])

D=pd.DataFrame(rows,columns=[
    "survivor","rank","band","file","x","y","nx","ny",
    "background_subtracted_flux","peak_pixel","aperture_snr_proxy",
    "centroid_x","centroid_y","background","background_robust_sigma",
    "aperture_pixels","background_pixels"
])

# Verify reproducibility against EXP-034C
M=C.merge(D,on=["survivor","rank","band"],suffixes=("_034C","_034F"))
M["flux_abs_diff"]=abs(M["background_subtracted_flux_034C"]-M["background_subtracted_flux_034F"])
M["peak_abs_diff"]=abs(M["peak_pixel_034C"]-M["peak_pixel_034F"])

D.to_csv(os.path.join(OUT,"exp034F_final_image_remeasurements.csv"),index=False)
M.to_csv(os.path.join(OUT,"exp034F_reproducibility_check.csv"),index=False)

print("EXP-034F FINAL IMAGE REMEASUREMENT")
print("Rows:",len(D))
print("All inside verified cutouts:",bool(
    ((D.x>=0)&(D.x<D.nx)&(D.y>=0)&(D.y<D.ny)).all()
))
print("Maximum flux difference vs EXP-034C:",
      M["flux_abs_diff"].max())
print("Maximum peak difference vs EXP-034C:",
      M["peak_abs_diff"].max())
print("\nMeasurements:")
print(D[["survivor","rank","band","background_subtracted_flux",
         "peak_pixel","aperture_snr_proxy","centroid_x","centroid_y",
         "background_robust_sigma"]].to_string(index=False))

print("\nSaved:")
print(os.path.join(OUT,"exp034F_final_image_remeasurements.csv"))
print(os.path.join(OUT,"exp034F_reproducibility_check.csv"))

EXP-034F FINAL IMAGE REMEASUREMENT
Rows: 16
All inside verified cutouts: True
Maximum flux difference vs EXP-034C: 146.4729461669922
Maximum peak difference vs EXP-034C: 1.4210854715202004e-14

Measurements:
 survivor  rank  band  background_subtracted_flux  peak_pixel  aperture_snr_proxy  centroid_x  centroid_y  background_robust_sigma
     1974     1     1                  589.809612  131.021973           24.270958   50.477491   49.759007                 4.366206
     1974     1     2                  190.639496  108.069237           12.009670   50.329542   49.831028                 5.229052
     1974     2     1                  602.586261   83.161407           24.478670   49.743630   49.809866                 3.798492
     1974     2     2                  222.420036   99.602722           13.954918   50.376956   50.456802                 6.789270
     1974     3     1                  545.363728   94.082916           23.353024   50.082971   50.124454                 3.849487
     1

In [5]:
import os,numpy as np,pandas as pd

ROOT="astronomy_exp005"
OUT=os.path.join(ROOT,"results")

C=pd.read_csv(os.path.join(OUT,"exp034C_image_photometry.csv"))
F=pd.read_csv(os.path.join(OUT,"exp034F_final_image_remeasurements.csv"))

M=C.merge(
    F,
    on=["survivor","rank","band"],
    suffixes=("_C","_F"),
    validate="one_to_one"
)

M["flux_diff"]=M["background_subtracted_flux_C"]-M["background_subtracted_flux_F"]
M["peak_diff"]=M["peak_pixel_C"]-M["peak_pixel_F"]

print("EXP-034F REPRODUCIBILITY DIAGNOSTIC")
print("C rows:",len(C),"F rows:",len(F),"merged:",len(M))
print("\nLargest flux discrepancies:")
print(
    M.loc[M["flux_diff"].abs().sort_values(ascending=False).index,
          ["survivor","rank","band","file_C",
           "background_subtracted_flux_C",
           "background_subtracted_flux_F",
           "flux_diff","peak_diff"]]
    .head(10).to_string(index=False)
)

print("\nExact matches:")
print("Flux exact:",int(np.isclose(M.flux_diff,0,atol=1e-10).sum()),"/",len(M))
print("Peak exact:",int(np.isclose(M.peak_diff,0,atol=1e-10).sum()),"/",len(M))

EXP-034F REPRODUCIBILITY DIAGNOSTIC
C rows: 16 F rows: 16 merged: 16

Largest flux discrepancies:
 survivor  rank  band                                                              file_C  background_subtracted_flux_C  background_subtracted_flux_F   flux_diff  peak_diff
     1978     3     2 astronomy_exp005/results/exp034_images/g1978_rank3_w2.fits                      0.078930                    146.551876 -146.472946        0.0
     1978     2     1 astronomy_exp005/results/exp034_images/g1978_rank2_w1.fits                    295.173283                    412.815899 -117.642616        0.0
     1974     4     1 astronomy_exp005/results/exp034_images/g1974_rank4_w1.fits                    552.043234                    642.426559  -90.383326        0.0
     1978     4     2 astronomy_exp005/results/exp034_images/g1978_rank4_w2.fits                    130.880219                    206.502357  -75.622139        0.0
     1974     3     2 astronomy_exp005/results/exp034_images/g1974_rank3_

In [ ]:
import os,glob,io,requests,warnings,numpy as np,pandas as pd
from astropy.io import fits
from astropy.wcs import WCS
from astropy.utils.exceptions import AstropyWarning

warnings.filterwarnings("ignore",category=AstropyWarning)

hits=glob.glob("/content/**/exp005_region_detections.csv",recursive=True)
if not hits: raise FileNotFoundError("exp005_region_detections.csv not found")
det_path=hits[0]
ROOT=os.path.dirname(os.path.dirname(det_path))
OUT=os.path.join(ROOT,"results")
IMG=os.path.join(OUT,"exp034_images")

DET=pd.read_csv(det_path)
C=pd.read_csv(os.path.join(OUT,"exp034C_image_photometry.csv"))

def phot(path,ra,dec):
    with fits.open(path,memmap=False) as h:
        d=np.squeeze(h[0].data).astype(float)
        w=WCS(h[0].header)
        x,y=w.world_to_pixel_values(float(ra),float(dec))
    yy,xx=np.indices(d.shape)
    rr=np.hypot(xx-x,yy-y)
    ap=(rr<=3)&np.isfinite(d)
    an=(rr>=6)&(rr<=12)&np.isfinite(d)
    if not ap.any() or not an.any(): return [np.nan]*9
    bg=np.median(d[an])
    bgmad=1.4826*np.median(np.abs(d[an]-bg))
    vals=d[ap]-bg
    flux=vals.sum()
    peak=np.max(d[ap])
    wgt=np.clip(vals,0,None)
    ys,xs=np.where(ap)
    sw=wgt.sum()
    cx=(xs*wgt).sum()/sw if sw>0 else np.nan
    cy=(ys*wgt).sum()/sw if sw>0 else np.nan
    snr=flux/(np.sqrt(np.sum(np.abs(vals)))+1e-9)
    return [x,y,flux,peak,snr,cx,cy,bg,bgmad]

# Recover exact influential observations from DET using survivor/rank ordering
survivors=[1974,1978]
infl=[]

for gid in survivors:
    g=DET[DET["group_id"]==gid].copy()
    g["z"]=(
        (g["w1mpro"]-g["w1mpro"].median())/
        (g["w1sigmpro"]+1e-9)
    ).abs()
    top=g.sort_values("z",ascending=False).head(4)
    top["rank"]=range(1,len(top)+1)
    infl.append(top)

I=pd.concat(infl,ignore_index=True)

# Exact influential frame keys
bad=set(zip(
    I["group_id"].astype(int),
    I["scan_id"].astype(str),
    I["frame_num"].astype(int)
))

# Measure influential W1 images from their OWN WCS
A=[]
for _,r in I.iterrows():
    gid=int(r["group_id"])
    rank=int(r["rank"])
    p=os.path.join(IMG,f"g{gid}_rank{rank}_w1.fits")
    if not os.path.exists(p): raise FileNotFoundError(p)
    v=phot(p,r["ra"],r["dec"])
    A.append([
        gid,rank,float(r["ra"]),float(r["dec"]),float(r["mjd"]),
        str(r["scan_id"]),int(r["frame_num"]),p,*v
    ])

A=pd.DataFrame(A,columns=[
    "survivor","rank","ra","dec","mjd","scan_id","frame_num","file",
    "x","y","flux","peak","snr","cx","cy","bg","bgmad"
])
A["type"]="INFLUENTIAL"

# Select one good-quality NORMAL observation per approximate epoch,
# explicitly excluding every influential frame
D=DET[DET["group_id"].isin(survivors)].copy()
D["epoch"]=(D["mjd"]//150).astype(int)

norm=[]
for gid in survivors:
    for ep,g in D[D["group_id"]==gid].groupby("epoch"):
        g=g[g["good_quality"].astype(bool)].copy()
        g=g[~g.apply(
            lambda r:(int(gid),str(r["scan_id"]),int(r["frame_num"])) in bad,
            axis=1
        )]
        if len(g)==0: continue
        med=g["w1mpro"].median()
        r=g.iloc[np.argmin(np.abs(g["w1mpro"]-med))]
        norm.append(r)

N=pd.DataFrame(norm)
if len(N)!=6:
    raise RuntimeError(f"Expected 6 normal observations, found {len(N)}")

# Download exact normal W1 frames
base="https://irsa.ipac.caltech.edu/ibe/data/wise/neowiser/p1bm_frm"
NR=[]

for _,r in N.iterrows():
    gid=int(r["group_id"])
    ep=int((r["mjd"])//150)
    sid=str(r["scan_id"])
    fr=int(r["frame_num"])

    q=("https://irsa.ipac.caltech.edu/ibe/search/wise/neowiser/p1bm_frm"
       f"?where=scan_id='{sid}'%20and%20frame_num={fr}%20and%20band=1&ct=CSV")
    resp=requests.get(q,timeout=30)
    resp.raise_for_status()
    m=pd.read_csv(io.StringIO(resp.text))
    if len(m)==0: raise RuntimeError(f"No metadata for {sid} frame {fr}")

    sg=str(m.iloc[0]["scangrp"])
    url=f"{base}/{sg}/{sid}/{fr:03d}/{sid}{fr:03d}-w1-int-1b.fits"

    p=os.path.join(OUT,f"exp034F_normal_g{gid}_e{ep}_w1.fits")
    z=requests.get(url,timeout=60)
    z.raise_for_status()
    if not z.content.startswith(b"SIMPLE"):
        raise RuntimeError(f"Invalid FITS: {url}")
    with open(p,"wb") as f: f.write(z.content)

    NR.append([
        gid,ep,float(r["ra"]),float(r["dec"]),float(r["mjd"]),
        sid,fr,sg,p
    ])

NR=pd.DataFrame(NR,columns=[
    "survivor","epoch","ra","dec","mjd",
    "scan_id","frame_num","scangrp","file"
])

# Measure normal images using EACH image's own WCS
B=[]
for _,r in NR.iterrows():
    v=phot(r["file"],r["ra"],r["dec"])
    B.append([
        int(r["survivor"]),int(r["epoch"]),float(r["ra"]),float(r["dec"]),
        float(r["mjd"]),r["scan_id"],int(r["frame_num"]),r["file"],*v
    ])

B=pd.DataFrame(B,columns=[
    "survivor","epoch","ra","dec","mjd","scan_id","frame_num","file",
    "x","y","flux","peak","snr","cx","cy","bg","bgmad"
])
B["type"]="NORMAL"

# Combined comparison
S=pd.concat([
    A[["survivor","rank","ra","dec","mjd","scan_id","frame_num","type",
       "x","y","flux","peak","snr","cx","cy","bg","bgmad"]],
    B.assign(rank=np.nan)[
        ["survivor","rank","ra","dec","mjd","scan_id","frame_num","type",
         "x","y","flux","peak","snr","cx","cy","bg","bgmad"]
    ]
],ignore_index=True)

summary=S.groupby(["survivor","type"]).agg(
    n=("flux","size"),
    median_flux=("flux","median"),
    median_peak=("peak","median"),
    median_snr=("snr","median"),
    median_bg=("bg","median"),
    median_bgmad=("bgmad","median")
).reset_index()

# Direct influential/normal ratios
cmp=[]
for gid in survivors:
    a=A[A["survivor"]==gid]
    b=B[B["survivor"]==gid]
    cmp.append([
        gid,
        a["flux"].median(),
        b["flux"].median(),
        a["flux"].median()/b["flux"].median() if b["flux"].median()!=0 else np.nan,
        a["peak"].median(),
        b["peak"].median(),
        a["snr"].median(),
        b["snr"].median()
    ])

cmp=pd.DataFrame(cmp,columns=[
    "survivor","influential_median_flux","normal_median_flux",
    "flux_ratio","influential_median_peak","normal_median_peak",
    "influential_median_snr","normal_median_snr"
])

S.to_csv(os.path.join(OUT,"exp034F_influential_vs_normal.csv"),index=False)
NR.to_csv(os.path.join(OUT,"exp034F_normal_observations.csv"),index=False)
summary.to_csv(os.path.join(OUT,"exp034F_final_summary.csv"),index=False)
cmp.to_csv(os.path.join(OUT,"exp034F_comparison.csv"),index=False)

print("EXP-034F CORRECTED COMPLETE")
print("Workspace:",ROOT)
print("\nNormal observations:")
print(NR[["survivor","epoch","mjd","scan_id","frame_num"]].to_string(index=False))
print("\nSummary:")
print(summary.to_string(index=False))
print("\nComparison:")
print(cmp.to_string(index=False))
print("\nSaved:",OUT)